# Full-length BATDiff on Colab (paper 120000 steps)

I use this notebook to run the public BATDiff method on one photograph: x4 super-resolution, published bicubic `x_ref`, no DIP.

The paper trains 120000 steps on an H100 with batch 16. A Colab T4 cannot hold a full 2K DIV2K photo at `dim=200`, so I use a 256x256 centre crop. That is the same size as my palm identity run, so the timing is comparable.

| What | Paper | This notebook |
|---|---|---|
| Training steps | 120000 | 120000 |
| Diffusion steps | 100 | 100 |
| A-trous levels | 6, wavelet `b3` | same |
| Scale | x4 | x4 |
| Batch (`ts`) | 16 | 4 (T4 memory) |
| Width (`dim`) | 200 | 200 |
| Image | full DIV2K | 256 crop |
| GPU | H100 | Tesla T4 |

20k steps on this 256 crop took about 5 hours (~4000 steps/hour).

| Run | Steps | T4 time (estimate) |
|---|---|---|
| Smoke test | 101 | 5–10 minutes |
| This notebook, from scratch | 120000 | about 28–36 hours |

Free Colab often stops around 12 hours, Colab Pro around 24 hours, so 120k will not finish in one sitting. I save a checkpoint every 10000 steps and continue from the newest `model-N.pt` if the session dies.

I use Chrome, set the runtime to T4 GPU, and upload one PNG such as `outputs/sanity/batdiff_hr_identity/0806_crop256.png`. I leave the tab open. I skip Google Drive here because it failed on Safari earlier.


In [ ]:
#@title Step 0 — check the runtime has a GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {total_gb:.1f} GB")
print(f"torch: {torch.__version__}")
print("Full 120k estimate on T4: about 28-36 hours (not one Colab session).")

In [ ]:
#@title Step 1 — clone BATDiff and install four packages
import os, subprocess, sys
from pathlib import Path

PROJECT_REPO = "https://github.com/yoyowuyogwrt-hue/3D-OCT-Image-SuperResolution-Benchmark"
BATDIFF_REPO = "https://github.com/MaryamHeidari-1994/BATDiff"

WORK = Path("/content")
BATDIFF = WORK / "BATDiff"
PROJECT = WORK / "project"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


if not BATDIFF.exists():
    sh(f"git clone --depth 1 {BATDIFF_REPO} {BATDIFF}")
if not PROJECT.exists():
    sh(f"git clone --depth 1 {PROJECT_REPO} {PROJECT}")

sh("pip install -q einops ftfy regex PyWavelets lpips")
sh(f"python {PROJECT}/scripts/batdiff_dip_patch.py --batdiff-root {BATDIFF}")
print("Patch applied. This run omits --xref_image (published bicubic x_ref).")

## Step 2 — Upload one photograph

I upload one PNG: `0806_crop256.png` (already 256) or a full DIV2K file such as `0801.png`, which I centre-crop to 256.

To continue after Colab stops, I also upload the newest `model-N.pt` (for example `model-3.pt` after 30k steps).


In [ ]:
#@title Step 2 — upload photo (and optional checkpoint)
from google.colab import files
from PIL import Image
from IPython.display import display
import io, shutil

CROP_SIZE = 256
SR_FACTOR = 4
DATA = WORK / "data" / "batdiff_full"
DATA.mkdir(parents=True, exist_ok=True)
UPLOADED_CKPT = None
LOAD_MILESTONE = 0

print("Upload one PNG. Optional: also upload model-N.pt to resume.")
uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("Upload a PNG.")

image = None
for raw_name, raw_bytes in uploaded.items():
    lower = raw_name.lower()
    if lower.endswith(".pt"):
        import re
        match = re.search(r"model-(\d+)", raw_name)
        if not match:
            raise ValueError(f"Checkpoint must be named model-N.pt, got {raw_name}")
        LOAD_MILESTONE = int(match.group(1))
        UPLOADED_CKPT = DATA / f"model-{LOAD_MILESTONE}.pt"
        UPLOADED_CKPT.write_bytes(raw_bytes)
        print(f"resume checkpoint: {raw_name} (milestone {LOAD_MILESTONE})")
    elif lower.endswith((".png", ".jpg", ".jpeg")):
        image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        print(f"image: {raw_name} {image.size[0]}x{image.size[1]}")

if image is None:
    raise FileNotFoundError("Missing a PNG photograph.")

width, height = image.size
if width < CROP_SIZE or height < CROP_SIZE:
    raise ValueError(f"Image {image.size} is smaller than {CROP_SIZE}.")
left = (width - CROP_SIZE) // 2
top = (height - CROP_SIZE) // 2
hr = image.crop((left, top, left + CROP_SIZE, top + CROP_SIZE))
HR = DATA / "hr.png"
hr.save(HR)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import numpy as np
import torch as torch_mod
from src.degradation import filtered_stride_downsample

hr_tensor = torch_mod.from_numpy(np.asarray(hr, dtype=np.float32) / 255.0)
hr_tensor = hr_tensor.permute(2, 0, 1).unsqueeze(0)
lr_tensor = filtered_stride_downsample(hr_tensor, SR_FACTOR)
lr_array = (lr_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1).numpy() * 255.0).round().astype("uint8")
lr = Image.fromarray(lr_array, mode="RGB")
LR = DATA / "lr.png"
lr.save(LR)
print(f"HR {hr.size[0]}x{hr.size[1]}  LR {lr.size[0]}x{lr.size[1]}  (blur then ::{SR_FACTOR})")
display(hr)
display(lr.resize(hr.size, Image.NEAREST))

In [ ]:
#@title Step 3 — settings (smoke first, then 120k)
SMOKE_TEST = True  #@param {type:"boolean"}

if SMOKE_TEST:
    DIM, TRAIN_STEPS, TIMESTEPS, TS, SAVE_EVERY = 200, 101, 100, 4, 101
    load_milestone = 0
else:
    DIM, TRAIN_STEPS, TIMESTEPS, TS, SAVE_EVERY = 200, 120000, 100, 4, 10000
    load_milestone = LOAD_MILESTONE

RESULTS = WORK / "results_batdiff_full"
RESULTS.mkdir(parents=True, exist_ok=True)
hours = TRAIN_STEPS / 4000.0

COMMON_FLAGS = (
    f"--mode train --image_name lr.png "
    f"--use_atrous --atrous_wavelet b3 --atrous_level 6 "
    f"--sr_factor {SR_FACTOR} --dim {DIM} --ts {TS} "
    f"--train_num_steps {TRAIN_STEPS} --timesteps {TIMESTEPS} "
    f"--save_and_sample_every {SAVE_EVERY} "
)
if load_milestone:
    COMMON_FLAGS += f"--load_milestone {load_milestone} "

print("SMOKE TEST" if SMOKE_TEST else "FULL 120k BATDiff x4")
print(f"  dim          {DIM}")
print(f"  ts           {TS}  (paper uses 16; 4 fits a T4)")
print(f"  train steps  {TRAIN_STEPS}")
print(f"  load         {load_milestone}")
print(f"  sr_factor    {SR_FACTOR}")
print(f"  T4 estimate  ~{hours:.0f} hours from step 0")
if not SMOKE_TEST:
    remaining = max(TRAIN_STEPS - load_milestone * SAVE_EVERY, 0) if load_milestone else TRAIN_STEPS
    if load_milestone:
        # milestone k was saved at k * SAVE_EVERY steps
        remaining = TRAIN_STEPS - load_milestone * SAVE_EVERY
        print(f"  remaining    ~{remaining / 4000.0:.0f} hours after milestone {load_milestone}")
    print("Keep this tab open. Do not close the Mac lid.")

In [ ]:
#@title Step 4 — train BATDiff and download the zip
import re, time


def find_final_image(scope_dir: Path) -> Path:
    candidates = list((scope_dir / "final_samples").glob("*.png"))
    if not candidates:
        raise FileNotFoundError(f"No samples under {scope_dir / 'final_samples'}")

    def scale_of(path: Path) -> int:
        match = re.search(r"_s(\d+)_", path.name)
        return int(match.group(1)) if match else -1

    finest = max(scale_of(p) for p in candidates)
    at_finest = [p for p in candidates if scale_of(p) == finest]
    return max(at_finest, key=lambda p: p.stat().st_mtime)


tag = "full"
scope_dir = RESULTS / tag / tag
scope_dir.mkdir(parents=True, exist_ok=True)
if load_milestone and UPLOADED_CKPT is not None:
    dest = scope_dir / f"model-{load_milestone}.pt"
    shutil.copy(UPLOADED_CKPT, dest)
    print("checkpoint copied to", dest)

flags = COMMON_FLAGS + (
    f"--scope {tag} --dataset_folder {DATA}/ --results_folder {RESULTS / tag} "
)
print("flags:", flags)

started = time.time()
process = subprocess.Popen(
    f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python main.py {flags}",
    shell=True, cwd=BATDIFF, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
for line in process.stdout:
    sys.stdout.write(line)
process.wait()
if process.returncode:
    raise RuntimeError(f"BATDiff failed with exit code {process.returncode}")

out = find_final_image(scope_dir)
print(f"\nfinished in {(time.time() - started) / 60:.1f} min -> {out.name}")
archive = shutil.make_archive(str(WORK / "batdiff_full_results"), "zip", RESULTS)
print("zip:", archive)
try:
    files.download(archive)
except Exception as error:
    print(f"(automatic download unavailable: {error}; use the Files pane)")

In [ ]:
#@title Step 5 — compare with the original crop
import matplotlib.pyplot as plt
from src.metrics.evaluate import evaluate

hr_np = np.asarray(Image.open(HR).convert("RGB"))
lr_np = Image.open(LR).convert("RGB")
bicubic = np.asarray(lr_np.resize((hr_np.shape[1], hr_np.shape[0]), Image.BICUBIC))
bat = np.asarray(Image.open(out).convert("RGB"))
if bat.shape != hr_np.shape:
    bat = np.asarray(Image.open(out).convert("RGB").resize((hr_np.shape[1], hr_np.shape[0]), Image.BICUBIC))

rows = []
for name, image in (("Bicubic upsample", bicubic), ("BATDiff", bat)):
    scores = evaluate(hr_np, image)
    rows.append((name, image, scores))
    print(f"{name}: PSNR={scores['PSNR']:.4f}  SSIM={scores['SSIM']:.4f}  LPIPS={scores['LPIPS']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.4))
axes[0].imshow(hr_np)
axes[0].set_title("HR crop")
axes[0].axis("off")
for ax, (name, image, scores) in zip(axes[1:], rows):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
    ax.text(
        0.5, -0.06,
        f"PSNR {scores['PSNR']:.2f}  SSIM {scores['SSIM']:.3f}  LPIPS {scores['LPIPS']:.3f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=8,
    )
fig.tight_layout()
figure_path = RESULTS / ("comparison_smoke.png" if SMOKE_TEST else "comparison.png")
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print("saved", figure_path)
if SMOKE_TEST:
    print("Smoke test only. Set SMOKE_TEST=False, Runtime -> Restart runtime, run from Step 0.")

## If Colab stops before 120k

I download the newest `model-N.pt` from the Files pane (`results_batdiff_full/full/full/`), start a new T4 runtime, and run from Step 0. In Step 2 I upload the same PNG and that checkpoint. I set `SMOKE_TEST = False` and run Step 3–4. The log should show `load N` and continue toward 120k.

`model-1.pt` is 10k steps, `model-2.pt` is 20k, … `model-12.pt` is 120k.

I do not load the palm identity `model-8.pt` here. That file is `sr_factor=1`. This notebook is x4 super-resolution.
